# Document-Aware Extraction with Llama Vision

Document-aware extraction notebook for processing invoices, receipts, and bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Document type detection - Automatically identifies invoice, receipt, or bank statement
- YAML-based prompt configuration - Static prompts for each document type
- V100 GPU optimization - ResilientGenerator with 6-tier OOM fallback system
- Universal field extraction - Consistent fields across all document types
- Memory management - Real-time fragmentation detection and cleanup
- Rich console interface - Beautiful formatting and comprehensive status reporting
- Production-ready error handling - Emergency model reload and CPU fallback
- Ground truth validation - Automated accuracy assessment and performance metrics

**Document Types Supported:**
- **Invoices** - Bills, quotes, estimates with line items and totals
- **Receipts** - Purchase confirmations with items and payment details
- **Bank Statements** - Transaction tables with dates and amounts

**Modern Architecture:**
- Modular V100-optimized extractor class
- External configuration files (YAML)
- Comprehensive memory monitoring
- Advanced error recovery systems

# Core Library Imports

Essential imports for Vision Language Model processing, image handling, and data manipulation.

In [ ]:
# Standard library imports
import warnings
from pathlib import Path

# Third-party imports
import pandas as pd
import torch
import yaml
from PIL import Image
from rich import print as rprint
from rich.console import Console
from rich.syntax import Syntax
from rich.table import Table
from transformers import AutoProcessor, MllamaForConditionalGeneration

# Local imports - GPU optimization utilities
from common.gpu_optimization import (
    comprehensive_memory_cleanup,
    configure_cuda_memory_allocation,
    optimize_model_for_v100,
)

warnings.filterwarnings('ignore')

rprint("[bold green]✅ Core libraries imported successfully[/bold green]")

# Configuration and Model Setup

In [ ]:
# Configuration for document type detection and extraction
DETECTION_PROMPT_FILE = "prompts/document_type_detection.yaml"  # Document type detection prompts
DETECTION_PROMPT_KEY = "detection"  # Which detection prompt to use

# Extraction prompt files for each document type
PROMPT_FILES = {
    "INVOICE": "prompts/invoice_extraction.yaml",
    "RECEIPT": "prompts/receipt_extraction.yaml",
    "BANK_STATEMENT": "prompts/bank_statement_extraction_original.yaml"
}

# Default prompt key for each document type
PROMPT_KEYS = {
    "INVOICE": "standard",
    "RECEIPT": "standard", 
    "BANK_STATEMENT": "flat"
}

# Configuration for model
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Test images - can be any document type
TEST_IMAGE = "evaluation_data/commbank_flat_simple.png"  # Bank statement
# TEST_IMAGE = "evaluation_data/image_001.png"  # Invoice
# TEST_IMAGE = "evaluation_data/image_002.png"  # Receipt
# TEST_IMAGE = "evaluation_data/commbank_flat_complex.png"  # Bank statement
# TEST_IMAGE = "evaluation_data/anz_statement_001.png"  # Bank statement

# Ground truth CSV for evaluation
GROUND_TRUTH_CSV = "evaluation_data/ground_truth.csv"

# V100 PRODUCTION CONFIGURATION - MODERN APPROACH
USE_QUANTIZATION = True   # Enable BitsAndBytesConfig (modern approach)
DEVICE_MAP = "auto"       # Automatic device mapping
MAX_NEW_TOKENS = 4000     # V100 OPTIMIZED - Prevents OOM
TORCH_DTYPE = "bfloat16"  # V100 COMPATIBLE - More efficient
LOW_CPU_MEM_USAGE = True  # MEMORY EFFICIENT - Essential for V100

# Initialize Rich console
console = Console()
rprint("[bold blue]🚀 Document-Aware Configuration loaded[/bold blue]")
rprint("[yellow]⚠️ V100 Production Mode: Modern BitsAndBytesConfig approach[/yellow]")
rprint(f"[cyan]🎯 TEST IMAGE: {TEST_IMAGE}[/cyan]")
rprint("[green]💡 Document type will be auto-detected[/green]")

# Validate the selected image exists
if Path(TEST_IMAGE).exists():
    rprint(f"[green]✅ Test image found: {Path(TEST_IMAGE).name}[/green]")
else:
    rprint(f"[red]❌ Test image not found: {TEST_IMAGE}[/red]")
    rprint("[yellow]💡 Update TEST_IMAGE path above[/yellow]")

# Image Selection & Validation

Select and validate the bank statement image for processing, with fallback to available images.

In [ ]:
# =============================================================================
# DOCUMENT IMAGE SELECTION & VALIDATION
# =============================================================================

from common.image_validator import validate_document_image

# Validate and display image information
IMAGE_PATH = validate_document_image(TEST_IMAGE)

# Model Loading & Validation

Load the Llama 3.2 Vision model with comprehensive validation and GPU optimization for V100.

In [ ]:
# =============================================================================
# MODEL LOADING & VALIDATION - V100 PRODUCTION OPTIMIZED
# =============================================================================

from common.model_loader import load_v100_model

# Load model with V100 optimizations
model, processor = load_v100_model(
    model_path=MODEL_PATH,
    use_quantization=USE_QUANTIZATION,
    device_map=DEVICE_MAP,
    max_new_tokens=MAX_NEW_TOKENS,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=LOW_CPU_MEM_USAGE
)

# Prompt Engineering

Define the optimized prompt for bank statement transaction extraction with clear visual guidance.

In [ ]:
# Import V100-optimized modules 
from common.ground_truth_helpers import get_expected_transaction_count
from common.llama_vision_table_extractor import LlamaVisionTableExtractor

rprint("[bold green]✅ V100-optimized modules imported successfully[/bold green]")

# =============================================================================
# V100 PRODUCTION INITIALIZATION
# =============================================================================

# Initialize the V100-optimized extractor with existing model and processor
rprint("[bold green]🚀 Initializing V100-optimized LlamaVisionTableExtractor...[/bold green]")

try:
    extractor = LlamaVisionTableExtractor(processor=processor, model=model)
    
    # Validate V100 optimization status
    if hasattr(extractor, 'resilient_generator'):
        rprint("[green]✅ ResilientGenerator initialized for V100 OOM protection[/green]")
    else:
        rprint("[red]❌ ResilientGenerator initialization failed[/red]")
        
    rprint("[green]✅ V100-optimized extractor ready for production use![/green]")
    rprint("[cyan]🎯 V100 Features Active:[/cyan]")
    rprint("[cyan]   • ResilientGenerator with 6-tier OOM fallback[/cyan]")
    rprint("[cyan]   • Memory fragmentation detection and cleanup[/cyan]") 
    rprint("[cyan]   • Individual processing for memory stability[/cyan]")
    rprint("[cyan]   • Emergency model reload capabilities[/cyan]")
    rprint("[cyan]   • CPU fallback as ultimate safety net[/cyan]")
    
except Exception as e:
    rprint(f"[red]❌ CRITICAL: V100 extractor initialization failed: {e}[/red]")
    raise

# Document Type Detection

Detect the type of document (invoice, receipt, or bank statement) before selecting the appropriate extraction prompt.

In [ ]:
# =============================================================================
# DOCUMENT TYPE DETECTION
# =============================================================================

from common.document_detector import detect_document_type

rprint("[bold blue]📋 Document Type Detection[/bold blue]")

# Detect document type using the extractor
DOCUMENT_TYPE = detect_document_type(
    image_path=IMAGE_PATH,
    extractor=extractor,
    detection_prompt_file=DETECTION_PROMPT_FILE,
    prompt_files=PROMPT_FILES,
    detection_prompt_key=DETECTION_PROMPT_KEY
)

console.rule("[bold green]Document Type Detection Complete[/bold green]")

# ============================================================================= 
# DOCUMENT-SPECIFIC PROMPT LOADING FROM YAML
# =============================================================================

from common.prompt_loader import load_document_prompt

rprint("[bold blue]📋 Loading Document-Specific Prompt from YAML[/bold blue]")

# Load document-specific prompt
EXTRACTION_PROMPT, prompt_name, prompt_description = load_document_prompt(
    prompt_files=PROMPT_FILES,
    prompt_keys=PROMPT_KEYS,
    document_type=DOCUMENT_TYPE
)

from common.ground_truth_helpers import get_expected_transaction_count
from common.llama_vision_table_extractor import LlamaVisionTableExtractor

rprint("[bold green]✅ V100-optimized modules imported successfully[/bold green]")


# Document-Aware Extraction

In [ ]:
# =============================================================================
# V100 PRODUCTION DEMONSTRATION - DOCUMENT-AWARE EXTRACTION
# =============================================================================

from common.result_display import display_extraction_test_results
from common.ground_truth_helpers import get_expected_transaction_count

# Test with the current image and document-specific prompt
if IMAGE_PATH:
    # Use the enhanced test_extraction method which includes V100 memory monitoring
    test_result = extractor.test_extraction(IMAGE_PATH, EXTRACTION_PROMPT)
    
    # Display comprehensive test results using the result display module
    display_extraction_test_results(
        test_result=test_result,
        document_type=DOCUMENT_TYPE,
        image_path=IMAGE_PATH,
        expected_count_func=get_expected_transaction_count,
        ground_truth_csv=GROUND_TRUTH_CSV
    )
else:
    rprint("[red]❌ No image path available for V100 testing[/red]")

# Visual Comaprison

In [ ]:
# =============================================================================
# VISUAL COMPARISON - DOCUMENT IMAGE DISPLAY
# =============================================================================

from IPython.display import Image as IPImage
from IPython.display import display

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    rprint(f"[cyan]📄 Displaying {DOCUMENT_TYPE} for visual comparison: {Path(IMAGE_PATH).name}[/cyan]")
    
    # Display the image
    display(IPImage(filename=IMAGE_PATH))
    
    rprint(f"[green]✅ Compare extraction results above with the visual {DOCUMENT_TYPE} image[/green]")
else:
    rprint(f"[red]❌ Document image not found: {IMAGE_PATH}[/red]")

# Field-Level Ground Truth Evaluation

Comprehensive field-by-field comparison between extracted data and ground truth values.

In [ ]:
# =============================================================================
# RESPONSE PREPROCESSING - Import from common module
# =============================================================================

from common.response_preprocessing import (
    clean_markdown_response,
    extract_statement_date_range,
    extract_transaction_data_from_table,
    map_bank_fields_to_universal,
    map_invoice_fields_to_universal,
    map_receipt_fields_to_universal,
    map_fields_to_universal,  # Universal mapping function
)

rprint("[green]✅ Response preprocessing functions imported from common module[/green]")

# Ground Truth Evaluation Summary 

In [ ]:
# =============================================================================
# FIELD-LEVEL GROUND TRUTH EVALUATION - DOCUMENT-AWARE
# =============================================================================

from common.evaluation_metrics import calculate_field_accuracy
from common.extraction_parser import parse_extraction_response

rprint("[bold cyan]📊 FIELD-LEVEL GROUND TRUTH EVALUATION[/bold cyan]")
console.rule("[bold blue]Comprehensive Document-Aware Field Evaluation[/bold blue]")

if IMAGE_PATH and 'test_result' in globals() and test_result.get('success'):
    # Get the raw response from the test
    raw_response = test_result['raw_result']['raw_response']
    
    # CLEAN THE RESPONSE BEFORE PARSING
    cleaned_response = clean_markdown_response(raw_response)
    
    # MAP DOCUMENT-SPECIFIC FIELDS TO UNIVERSAL FIELD NAMES FOR EVALUATION
    mapped_response = map_fields_to_universal(cleaned_response, DOCUMENT_TYPE)
    
    image_filename = Path(IMAGE_PATH).name
    
    rprint(f"[yellow]🔍 Evaluating extraction for: {image_filename}[/yellow]")
    rprint(f"[dim]Document Type: {DOCUMENT_TYPE}[/dim]")
    rprint(f"[dim]Using {DOCUMENT_TYPE.lower()}-specific extraction with universal field mapping for evaluation[/dim]")
    
    # Parse the MAPPED extraction response to get field-level data
    extracted_fields = parse_extraction_response(mapped_response)
    
    # Load ground truth for this specific image
    try:
        ground_truth_df = pd.read_csv(GROUND_TRUTH_CSV)
        image_row = ground_truth_df[ground_truth_df['image_file'] == image_filename]
        
        if not image_row.empty:
            ground_truth = image_row.iloc[0].to_dict()
            
            rprint(f"[green]✅ Ground truth loaded for {image_filename}[/green]")
            rprint(f"[dim]Found {len([v for v in ground_truth.values() if v != 'NOT_FOUND' and pd.notna(v)])} ground truth fields[/dim]")
            
            # Create field comparison table
            comparison_table = Table(title=f"📋 Field-by-Field Comparison ({DOCUMENT_TYPE}→Universal Mapping)", border_style="green")
            comparison_table.add_column("Status", style="bold", width=8)
            comparison_table.add_column("Field", style="cyan", width=25)
            comparison_table.add_column("Extracted", style="yellow", max_width=40)
            comparison_table.add_column("Ground Truth", style="magenta", max_width=40)
            comparison_table.add_column("Score", style="blue", justify="right")
            
            # Track metrics
            total_fields = 0
            fields_found = 0
            exact_matches = 0
            partial_matches = 0
            field_scores = {}
            
            # Document type specific fields to focus on
            if DOCUMENT_TYPE == "BANK_STATEMENT":
                focus_fields = ["DOCUMENT_TYPE", "STATEMENT_DATE_RANGE", "LINE_ITEM_DESCRIPTIONS", 
                               "TRANSACTION_DATES", "TRANSACTION_AMOUNTS_PAID"]
            elif DOCUMENT_TYPE == "INVOICE":
                focus_fields = ["DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "INVOICE_DATE",
                               "LINE_ITEM_DESCRIPTIONS", "GST_AMOUNT", "TOTAL_AMOUNT"]
            elif DOCUMENT_TYPE == "RECEIPT":
                focus_fields = ["DOCUMENT_TYPE", "SUPPLIER_NAME", "INVOICE_DATE",
                               "LINE_ITEM_DESCRIPTIONS", "TOTAL_AMOUNT"]
            else:
                focus_fields = []
            
            # Compare each field
            for field_name in ground_truth.keys():
                if field_name == 'image_file':  # Skip the image filename column
                    continue
                    
                ground_value = ground_truth.get(field_name, 'NOT_FOUND')
                extracted_value = extracted_fields.get(field_name, 'NOT_FOUND')
                
                # Convert pandas NaN to NOT_FOUND
                if pd.isna(ground_value):
                    ground_value = 'NOT_FOUND'
                if pd.isna(extracted_value):
                    extracted_value = 'NOT_FOUND'
                    
                # Skip if both are NOT_FOUND (field not applicable)
                if str(ground_value) == 'NOT_FOUND' and str(extracted_value) == 'NOT_FOUND':
                    continue
                    
                total_fields += 1
                
                # Calculate field accuracy
                accuracy = calculate_field_accuracy(
                    extracted_value, 
                    ground_value, 
                    field_name,
                    debug=False
                )
                
                field_scores[field_name] = accuracy
                
                # Determine status
                if accuracy == 1.0:
                    status = "✅"
                    exact_matches += 1
                    if str(extracted_value) != 'NOT_FOUND':
                        fields_found += 1
                elif accuracy >= 0.8:
                    status = "≈"
                    partial_matches += 1
                    if str(extracted_value) != 'NOT_FOUND':
                        fields_found += 1
                else:
                    status = "❌"
                    if str(extracted_value) != 'NOT_FOUND':
                        fields_found += 1
                
                # Highlight focus fields for this document type
                if field_name in focus_fields:
                    field_name = f"🔑 {field_name}"
                
                # Truncate long values for display
                extracted_display = str(extracted_value)[:38] + ("..." if len(str(extracted_value)) > 38 else "")
                ground_display = str(ground_value)[:38] + ("..." if len(str(ground_value)) > 38 else "")
                
                # Add row to table
                comparison_table.add_row(
                    status,
                    field_name,
                    extracted_display,
                    ground_display,
                    f"{accuracy:.2f}"
                )
            
            # Display the comparison table
            console.print(comparison_table)
            
            # Calculate overall metrics
            overall_accuracy = sum(field_scores.values()) / len(field_scores) if field_scores else 0
            
            # Summary statistics table
            summary_table = Table(title=f"📈 Evaluation Summary ({DOCUMENT_TYPE} Extraction)", border_style="blue")
            summary_table.add_column("Metric", style="cyan")
            summary_table.add_column("Value", style="magenta")
            summary_table.add_column("Percentage", style="yellow")
            
            summary_table.add_row(
                "Document Type",
                DOCUMENT_TYPE,
                "-"
            )
            summary_table.add_row(
                "Total Fields Evaluated",
                str(total_fields),
                "100%"
            )
            summary_table.add_row(
                "Fields Found",
                str(fields_found),
                f"{(fields_found/total_fields*100):.1f}%" if total_fields > 0 else "0%"
            )
            summary_table.add_row(
                "Exact Matches",
                str(exact_matches),
                f"{(exact_matches/total_fields*100):.1f}%" if total_fields > 0 else "0%"
            )
            summary_table.add_row(
                "Partial Matches (≥0.8)",
                str(partial_matches),
                f"{(partial_matches/total_fields*100):.1f}%" if total_fields > 0 else "0%"
            )
            summary_table.add_row(
                "Overall Accuracy",
                f"{overall_accuracy:.3f}",
                f"{(overall_accuracy*100):.1f}%"
            )
            
            console.print(summary_table)
            
            # Document type detection accuracy
            if 'DOCUMENT_TYPE' in extracted_fields:
                extracted_doc_type = extracted_fields['DOCUMENT_TYPE']
                ground_doc_type = ground_truth.get('DOCUMENT_TYPE', 'NOT_FOUND')
                
                if extracted_doc_type != 'NOT_FOUND' and ground_doc_type != 'NOT_FOUND':
                    doc_type_match = extracted_doc_type.upper() == str(ground_doc_type).upper()
                    rprint("\n[bold]Document Type Detection:[/bold]")
                    rprint(f"  Extracted: {extracted_doc_type}")
                    rprint(f"  Ground Truth: {ground_doc_type}")
                    rprint(f"  Status: {'✅ Correct' if doc_type_match else '❌ Incorrect'}")
            
            # Performance assessment
            rprint("\n[bold cyan]📊 Performance Assessment:[/bold cyan]")
            rprint(f"[dim]Note: Using {DOCUMENT_TYPE.lower()}-specific extraction with field mapping[/dim]")
            if overall_accuracy >= 0.95:
                rprint("[bold green]🎉 EXCELLENT - Production Ready (≥95% accuracy)[/bold green]")
            elif overall_accuracy >= 0.85:
                rprint("[yellow]✅ GOOD - Minor improvements needed (85-94% accuracy)[/yellow]")
            elif overall_accuracy >= 0.75:
                rprint("[yellow]⚠️ FAIR - Significant improvements needed (75-84% accuracy)[/yellow]")
            else:
                rprint("[red]❌ POOR - Major improvements required (<75% accuracy)[/red]")
                
            # Processing performance
            if 'processing_time' in test_result:
                proc_time = test_result['processing_time']
                rprint(f"\n⏱️  Processing Time: {proc_time:.2f}s")
                if proc_time < 5:
                    rprint("[green]  Speed: Excellent (<5s)[/green]")
                elif proc_time < 10:
                    rprint("[yellow]  Speed: Good (5-10s)[/yellow]")
                else:
                    rprint("[red]  Speed: Needs optimization (>10s)[/red]")
                    
        else:
            rprint(f"[red]❌ No ground truth found for {image_filename}[/red]")
            rprint("[yellow]💡 Make sure the image filename matches exactly in ground_truth.csv[/yellow]")
            
    except Exception as e:
        rprint(f"[red]❌ Error loading ground truth: {e}[/red]")
        import traceback
        traceback.print_exc()
        
else:
    rprint("[yellow]⚠️ No extraction results available for evaluation[/yellow]")
    rprint("[dim]Run the extraction test in the cell above first[/dim]")

console.rule("[bold green]Field-Level Evaluation Complete[/bold green]")